# OIBSIP — Data Analytics Level 1 — EDA on Retail Sales Data

**Intern:** Dinesh Desale  
**Track:** Data Analytics  
**Task:** EDA on Retail Sales Data

## Objective
Perform exploratory data analysis on retail sales data to identify sales trends, customer behaviour patterns, product performance and actionable business insights.

### Required task coverage
- Dataset inspection: shape, data types, missing values
- Descriptive statistics
- Monthly and quarterly sales trends
- Customer demographics: age groups and gender
- Top 10 products and revenue by category
- Correlation heatmap
- One additional visualization
- Written observations and at least 3 actionable recommendations


## 1. Import libraries and load the dataset

The notebook downloads the dataset automatically if it is not already present in `data/sales_data.csv`.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

DATA_URL = "https://gist.githubusercontent.com/summerofgeorge/33dbb09adbdea670c31ee2c4fc9f2617/raw/32d15ca6e8989f74b6a3fa1fef317f394efd48f4/sales_data.csv"
LOCAL_PATH = 'data/sales_data.csv'

os.makedirs('data', exist_ok=True)

if not os.path.exists(LOCAL_PATH):
    df = pd.read_csv(DATA_URL)
    df.to_csv(LOCAL_PATH, index=False)
else:
    df = pd.read_csv(LOCAL_PATH)

df.head()

## 2. Initial inspection

In [ ]:
print('Shape:', df.shape)
print('\nData types:')
display(df.dtypes.to_frame('dtype'))

print('\nMissing values:')
display(df.isnull().sum().to_frame('missing_values'))

print('\nDuplicate rows:', df.duplicated().sum())

### Observation
Check the output above for the number of records, columns, missing values and duplicates before continuing.

In [ ]:
df['order_date'] = pd.to_datetime(df['order_date'], errors='coerce')
df['revenue'] = df['unit_price'] * df['quantity']

numeric_cols = ['unit_price', 'quantity', 'customer_age', 'revenue']
df[numeric_cols].describe().T

## 3. Descriptive statistics

The table above provides count, mean, standard deviation, minimum, quartiles and maximum for the main numerical variables.

## 4. Monthly sales trend

In [ ]:
monthly_sales = (df.set_index('order_date')
                 .resample('MS')['revenue']
                 .sum()
                 .reset_index())

plt.figure(figsize=(12,5))
sns.lineplot(data=monthly_sales, x='order_date', y='revenue', marker='o')
plt.title('Monthly Revenue Trend')
plt.xlabel('Month')
plt.ylabel('Revenue')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### Observation
Use the chart to identify months with unusually high or low revenue and consider whether promotions, seasonality or product demand could explain the pattern.

## 5. Quarterly sales trend

In [ ]:
quarterly_sales = (df.set_index('order_date')
                   .resample('QS')['revenue']
                   .sum()
                   .reset_index())

quarterly_sales['quarter'] = quarterly_sales['order_date'].dt.to_period('Q').astype(str)

plt.figure(figsize=(10,5))
sns.barplot(data=quarterly_sales, x='quarter', y='revenue')
plt.title('Quarterly Revenue Trend')
plt.xlabel('Quarter')
plt.ylabel('Revenue')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

display(quarterly_sales)

### Observation
Compare the quarterly totals to determine which periods contributed the most revenue.

## 6. Customer demographics — age groups

In [ ]:
bins = [0, 18, 25, 35, 45, 55, 65, 100]
labels = ['<18', '18-25', '26-35', '36-45', '46-55', '56-65', '66+']
df['age_group'] = pd.cut(df['customer_age'], bins=bins, labels=labels, right=True, include_lowest=True)

age_counts = df['age_group'].value_counts().sort_index()
plt.figure(figsize=(10,5))
sns.barplot(x=age_counts.index, y=age_counts.values)
plt.title('Customer Distribution by Age Group')
plt.xlabel('Age Group')
plt.ylabel('Number of Transactions')
plt.tight_layout()
plt.show()

display(age_counts.to_frame('transactions'))

### Observation
Identify the age group contributing the largest number of transactions. This can help define the target audience for marketing campaigns.

## 7. Gender breakdown

In [ ]:
gender_counts = df['customer_gender'].value_counts()

plt.figure(figsize=(7,5))
sns.barplot(x=gender_counts.index, y=gender_counts.values)
plt.title('Transactions by Customer Gender')
plt.xlabel('Gender')
plt.ylabel('Number of Transactions')
plt.tight_layout()
plt.show()

display(gender_counts.to_frame('transactions'))

### Observation
Compare transaction volume across gender groups. Avoid treating transaction count alone as customer value; revenue should also be considered.

## 8. Top 10 best-selling products

In [ ]:
top_products = (df.groupby('product_name', as_index=False)['quantity']
                .sum()
                .sort_values('quantity', ascending=False)
                .head(10))

plt.figure(figsize=(10,6))
sns.barplot(data=top_products, x='quantity', y='product_name')
plt.title('Top 10 Best-Selling Products by Quantity')
plt.xlabel('Units Sold')
plt.ylabel('Product')
plt.tight_layout()
plt.show()

display(top_products)

### Observation
The highest-volume products are candidates for stock prioritisation, bundles and cross-selling.

## 9. Revenue by product category

In [ ]:
category_revenue = (df.groupby('category', as_index=False)['revenue']
                    .sum()
                    .sort_values('revenue', ascending=False))

plt.figure(figsize=(9,5))
sns.barplot(data=category_revenue, x='category', y='revenue')
plt.title('Revenue by Product Category')
plt.xlabel('Category')
plt.ylabel('Revenue')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

display(category_revenue)

### Observation
The highest-revenue category should receive attention in inventory planning, promotions and product assortment decisions.

## 10. Correlation heatmap

In [ ]:
corr_cols = ['unit_price', 'quantity', 'customer_age', 'revenue']
corr = df[corr_cols].corr()

plt.figure(figsize=(8,6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='Blues', square=True)
plt.title('Correlation Matrix')
plt.tight_layout()
plt.show()

### Observation
Correlation measures linear association, not causation. Use the heatmap to identify which numerical variables move together most strongly.

## 11. Additional visualization — Revenue by age group and gender

In [ ]:
age_gender_revenue = (df.groupby(['age_group', 'customer_gender'], observed=False)['revenue']
                       .sum()
                       .reset_index())

plt.figure(figsize=(12,6))
sns.barplot(data=age_gender_revenue, x='age_group', y='revenue', hue='customer_gender')
plt.title('Revenue by Age Group and Gender')
plt.xlabel('Age Group')
plt.ylabel('Revenue')
plt.tight_layout()
plt.show()

### Observation
This view helps identify demographic segments that contribute strongly to revenue, which is more informative than transaction count alone.

## 12. Key business metrics


In [ ]:
total_revenue = df['revenue'].sum()
total_units = df['quantity'].sum()
unique_customers = df['customer_id'].nunique()
avg_order_value = df['revenue'].mean()

kpis = pd.DataFrame({
    'Metric': ['Total Revenue', 'Total Units Sold', 'Unique Customers', 'Average Transaction Value'],
    'Value': [total_revenue, total_units, unique_customers, avg_order_value]
})
display(kpis)

## 13. Actionable business recommendations

1. **Prioritise high-volume products:** Keep the top-selling products adequately stocked and consider bundles or cross-selling offers.
2. **Focus category strategy on revenue leaders:** Allocate inventory and promotional budget according to the categories generating the highest revenue.
3. **Use demographic targeting:** Tailor campaigns to the strongest age/gender revenue segments while testing campaigns across other segments.
4. **Plan around seasonal trends:** Use monthly and quarterly patterns to prepare inventory and promotions before high-demand periods.

## Conclusion
This EDA converts transaction-level retail data into sales, customer and product insights. The charts and KPIs provide a foundation for inventory planning, customer targeting and revenue-focused marketing decisions.

## Reproducibility
- Python 3.x
- pandas
- numpy
- matplotlib
- seaborn
- Jupyter Notebook / Google Colab

Dataset source is documented in the README. Do not claim findings until all notebook cells have been executed successfully.